# DI-Adapter Cached-Feature Full Training


In [ ]:
GITHUB_REPO_URL = "https://github.com/huyvanzzz/Group_HND_car.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = False

# Optional: restore a prebuilt feature cache from Hugging Face instead of running prepare-features.
# Example: CACHE_HF_REPO_ID = "your-user/cache"
CACHE_HF_REPO_ID = "huyvanzzz/cache"
CACHE_HF_REPO_TYPE = "dataset"

# Optional: restore an align checkpoint before finetune.
# Example: ALIGN_CHECKPOINT_HF_REPO_ID = "your-user/your-checkpoint-repo"
ALIGN_CHECKPOINT_HF_REPO_ID = ""
ALIGN_CHECKPOINT_HF_REPO_TYPE = "model"
ALIGN_CHECKPOINT_FILENAME = "align_best.pt"

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = "1" if USE_SAFE_CONFIG else "0"
os.environ["CACHE_HF_REPO_ID"] = CACHE_HF_REPO_ID
os.environ["CACHE_HF_REPO_TYPE"] = CACHE_HF_REPO_TYPE
os.environ["ALIGN_CHECKPOINT_HF_REPO_ID"] = ALIGN_CHECKPOINT_HF_REPO_ID
os.environ["ALIGN_CHECKPOINT_HF_REPO_TYPE"] = ALIGN_CHECKPOINT_HF_REPO_TYPE
os.environ["ALIGN_CHECKPOINT_FILENAME"] = ALIGN_CHECKPOINT_FILENAME
print({
    "repo": GITHUB_REPO_URL,
    "branch": GITHUB_BRANCH,
    "safe_config": USE_SAFE_CONFIG,
    "cache_hf_repo": CACHE_HF_REPO_ID,
    "align_checkpoint_hf_repo": ALIGN_CHECKPOINT_HF_REPO_ID,
    "align_checkpoint_filename": ALIGN_CHECKPOINT_FILENAME,
})


In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving; fi
cd Efficient_VLM_For_Autonomous_Driving
git checkout "$GITHUB_BRANCH"
git pull --ff-only
pip install -e .
pip install pycocoevalcap

In [ ]:
import os

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        print("HF_TOKEN secret not loaded:", type(exc).__name__)

print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
%%bash
nvidia-smi || true
python - <<'PY'
import torch
print({'cuda': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count()})
PY


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; fi
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full --debug --debug-samples 1


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter; fi
if [ -n "$CACHE_HF_REPO_ID" ]; then
  echo "Restoring feature cache from Hugging Face: $CACHE_HF_REPO_ID"
  rm -rf "$PROFILE/cache"
  mkdir -p "$PROFILE/cache"
  hf download "$CACHE_HF_REPO_ID" --repo-type "$CACHE_HF_REPO_TYPE" --local-dir "$PROFILE/cache"
  echo "Restored cache files:"
  find "$PROFILE/cache" -maxdepth 2 -type f | head -50
else
  echo "CACHE_HF_REPO_ID is empty; building full feature cache now."
  python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full --debug --debug-samples 1
fi


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; fi
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter; fi
RESUME=""
if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then RESUME="--resume $PROFILE/checkpoints/align_latest.pt"; fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align $RESUME --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe
else
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter
fi
mkdir -p "$PROFILE/checkpoints"
ALIGN_TARGET="$PROFILE/checkpoints/$ALIGN_CHECKPOINT_FILENAME"
if [ -f "$ALIGN_TARGET" ]; then
  echo "Align checkpoint already exists: $ALIGN_TARGET"
elif [ -n "$ALIGN_CHECKPOINT_HF_REPO_ID" ]; then
  echo "Restoring align checkpoint from Hugging Face: $ALIGN_CHECKPOINT_HF_REPO_ID/$ALIGN_CHECKPOINT_FILENAME"
  hf download "$ALIGN_CHECKPOINT_HF_REPO_ID" "$ALIGN_CHECKPOINT_FILENAME" --repo-type "$ALIGN_CHECKPOINT_HF_REPO_TYPE" --local-dir "$PROFILE/checkpoints"
else
  echo "ALIGN_CHECKPOINT_HF_REPO_ID is empty; expecting align checkpoint from the align training cell."
fi
ls -lh "$PROFILE/checkpoints" || true


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter
fi
python -m efficient_vlm_ad verify-align-checkpoint \
  --config "$CONFIG" \
  --checkpoint "$PROFILE/checkpoints/align_best.pt" \
  --debug \
  --debug-samples 1 \
  --debug-numerics
cat "$PROFILE/debug/align_checkpoint_verify.json"


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter; fi
if [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then RESUME="$PROFILE/checkpoints/finetune_latest.pt"; else RESUME="$PROFILE/checkpoints/align_best.pt"; fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$RESUME" --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter; fi
CHECKPOINT="$PROFILE/checkpoints/best_model.pt"
if [ ! -f "$CHECKPOINT" ]; then CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"; fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" --batch-size 16
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT"


In [ ]:
%%bash
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_di_adapter; fi
ls -lh "$PROFILE/checkpoints" || true
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
tail -n 20 "$PROFILE/debug/eval_progress.jsonl" || true
tail -n 20 "$PROFILE/debug/benchmark_progress.jsonl" || true
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/benchmark.json" || true
